# Silver: `patient_information`

**Reads:** `bronze.patient_information` (65,728 rows, all columns loaded as strings)

**Writes:** `silver.patient_information` (64,353 rows × 38 columns, standardized column names)

### What this notebook does

This notebook cleans and standardizes the Bronze patient information table into a case-level Silver table.

The main steps are:
- standardize column names and data types
- remove duplicate case records
- validate and repair OR timestamps where needed
- calculate OR and anesthesia durations
- standardize procedure names
- keep flags for records that were repaired or changed

### Validation approach

OR entry and exit times come from the OR system, while anesthesia start and stop times come from a separate source.

Because both describe the same case from different systems, the anesthesia timestamps are used to cross-check unusual OR durations and support timestamp repairs where the pattern is clear.

In [ ]:
b_pi = spark.table("bronze.patient_information")
print("rows=",b_pi.count(),"*" "columns=",len(b_pi.columns))

## 1. Standardise names, trim values, drop identical rows

Column names are converted to lowercase and whitespace is removed from all string values.

**Result:** 65,728 → 64,362 rows. **1,366 duplicate rows removed.**

In [ ]:
from pyspark.sql.functions import col, trim

# Standardize column names
b_pi = b_pi.toDF(*[c.strip().lower() for c in b_pi.columns])

# Trim column values
pi_trim = b_pi.select([trim(col(c)).alias(c) for c in b_pi.columns])

# Remove exact duplicates
pi_distinct = pi_trim.distinct()

print(pi_distinct.count(),len(pi_distinct.columns))

## 2. Check duplicate `log_id` values

After removing identical rows, some duplicate `log_id` values still remain.

Since `log_id` is used to identify a surgical case, these duplicates are checked before removing any records.

In [ ]:
dupe_ids = (pi_distinct.groupBy("log_id").count().filter("count > 1"))
#display(dupe_ids)

dupe_rows=(pi_distinct.join(dupe_ids.select("log_id"),"log_id","left_semi" ).orderBy("log_id"))
#display(dupe_rows)

### Resolving duplicate `log_id`

Most duplicate `log_id` records are identical. The remaining conflicts were checked and had the same OR duration.

For duplicate IDs, one record is kept using a consistent row-based ordering so the same record is selected each time.

**Result:** 64,362 → 64,354 rows. `log_id` is now unique.

In [ ]:
from pyspark.sql.functions import row_number, to_json, struct
from pyspark.sql.window import Window

# review to handle duplicates
w = Window.partitionBy("log_id").orderBy(to_json(struct(*pi_distinct.columns)))

si = (pi_distinct.withColumn("rn", row_number().over(w)).filter("rn = 1").drop("rn"))

print("shape:",si.count(), len(si.columns))

## 3. Flag scientific-notation `log_id`

Some `log_id` and `mrn` values appear in scientific notation, which may have caused loss of digits in the original identifiers.

These does not affect the timestamp or duration analysis. They are flagged for data quality tracking and to avoid using the affected identifiers for future joins.

**Result:** 39 `log_id` and 37 `mrn` values flagged.

In [ ]:
from pyspark.sql.functions import col

SCI = r"^[0-9]+\.?[0-9]*E\+[0-9]+$"

si_flagged = (si
    .withColumn("log_id_corrupt", col("LOG_ID").rlike(SCI))
    .withColumn("mrn_corrupt",    col("MRN").rlike(SCI)))

si_flagged.selectExpr(
    "SUM(CASE WHEN log_id_corrupt THEN 1 ELSE 0 END) AS log_id_flag",
    "SUM(CASE WHEN mrn_corrupt    THEN 1 ELSE 0 END) AS mrn_flag"
).show()

## 4. Standardising the timestamp columns

**The published format uses a two-digit year: `M/d/yy H:mm`.**

Zero parse failures across all four columns.

In [ ]:
from pyspark.sql.functions import to_timestamp

si_ts = (si_flagged
    .withColumn("in_or",  to_timestamp("in_or_dttm",  "M/d/yy H:mm"))
    .withColumn("out_or", to_timestamp("out_or_dttm", "M/d/yy H:mm"))
    .withColumn("an_start",to_timestamp("an_start_datetime", "M/d/yy H:mm"))
    .withColumn("an_stop", to_timestamp("an_stop_datetime", "M/d/yy H:mm")))

## 5. Exploring the key feature columns

In [ ]:
si_ts.selectExpr(
    "sum(case when in_or is null and out_or is not null then 1 else 0 end) as blank_in_or",
    "sum(case when in_or is not null and out_or is null then 1 else 0 end) as blank_out_or",
    "SUM(CASE WHEN in_or IS NULL AND out_or IS NULL THEN 1 ELSE 0 END) AS both_missing",
    "SUM(CASE WHEN in_or IS NULL OR out_or IS NULL THEN 1 ELSE 0 END) AS not_usable_data",
    "count(*) as totalrecords"
).show()

As of now `6,492` records are missing either of `in_or` or `out_or`. These records cannot be used for OR duration analysis.

## 6. Finding the data distribution

`in_or & out_or` specifies duration of **room occupancy, not surgical time**

In [ ]:
from pyspark.sql.functions import expr

si_dur = si_ts.withColumn("or_duration_min", expr("timestampdiff(Minute, in_or , out_or)"))

si_dur.selectExpr(
  "COUNT(or_duration_min) AS notnullrows",
  "SUM(CASE WHEN or_duration_min <= 0  THEN 1 ELSE 0 END) AS non_positive",
  "SUM(CASE WHEN or_duration_min < 10  THEN 1 ELSE 0 END) AS under_10",
  "SUM(CASE WHEN or_duration_min > 720 THEN 1 ELSE 0 END) AS over_720",
  "MIN(or_duration_min) AS min_dur",
  "MAX(or_duration_min) AS max_dur",
  "percentile(or_duration_min, 0.01) AS p01",
  "percentile(or_duration_min, 0.50) AS median",
  "percentile(or_duration_min, 0.99) AS p99"
).show()

- The `non_positive` record is invalid because `out_or` occurs before `in_or`.
- Both timestamp ends need to be reviewed before deciding on the repair.

### The single non-positive duration


In [ ]:
si_dur.filter("or_duration_min<=0").selectExpr("log_id","in_or","out_or","an_start", "an_stop","primary_procedure_nm").show()

Reading all four timestamps together shows exactly which end is wrong:

The an_stop and out_or are closely aligned, we will try to establish if there is relation between these two fields.

## 7. Checking anesthesia duration

`anes_duration_min` is calculated from the anesthesia timestamps and can be used to cross-check the OR duration.

There are **57,029** cases with an anesthesia duration. Cases without one cannot be compared against the OR timestamps.

No non-positive anesthesia durations were found, which gives us a reliable reference for comparing the OR durations.

In [ ]:
si_an=si_dur.withColumn("anes_duration_min", expr("timestampdiff(Minute, an_start, an_stop)"))

si_an.selectExpr(
    "COUNT(*) as scannedrows",
    "COUNT(anes_duration_min) AS notnullrows",
    "SUM(CASE WHEN anes_duration_min <= 0 THEN 1 ELSE 0 END) AS non_positive",
    "MIN(anes_duration_min) AS min_dur",
    "MAX(anes_duration_min) AS max_dur",
    "percentile(anes_duration_min, 0.5) AS median"
).show()

## 8. Relation between anesthesia start and OR start time

Before anesthesia can be used to correct anything, the normal relationship between the two
clocks needs to be measured first. This is the cell that licenses every repair that follows.

In [ ]:
#relation or gap between anes_start & in_or

si_an.selectExpr(
  "COUNT(CASE WHEN in_or IS NOT NULL AND an_start IS NOT NULL THEN 1 END) AS rows_checked",
  "percentile(ABS(timestampdiff(Minute, in_or, an_start)), 0.50) AS median_gap",
  "percentile(ABS(timestampdiff(Minute, in_or, an_start)), 0.95) AS median_gap",
  "percentile(ABS(timestampdiff(Minute, in_or, an_start)), 0.99) AS median_gap",
).show()

Across 55,815 cases, the median gap between `an_start` and `in_or` is **0 minutes**. The p95 is **2 minutes**, meaning 95% of cases are within 2 minutes, while the p99 of **10 minutes** means 99% are within 10 minutes.

Since the two timestamps match exactly for about half of the cases and remain close for most records, `an_start` can be used as a reasonable replacement when `in_or` is clearly incorrect.

In [ ]:
si_an.selectExpr(
    "COUNT(CASE WHEN out_or IS NOT NULL AND an_stop IS NOT NULL THEN 1 END) AS rows_checked",
    "percentile(ABS(timestampdiff(MINUTE, an_stop, out_or)), 0.50) AS median_gap",
    "percentile(ABS(timestampdiff(MINUTE, an_stop, out_or)), 0.95) AS p95_gap",
    "percentile(ABS(timestampdiff(MINUTE, an_stop, out_or)), 0.99) AS p99_gap"
).show()

The gap between out_or and an_stop is larger. The median gap is 8 minutes, while p95 is 18 minutes and p99 is 33 minutes. This means 95% of cases are within 18 minutes and 99% are within 33 minutes.

This shows that anesthesia care generally continues after the patient leaves the OR.

### Checking `an_start` as a replacement for `in_or`

Since `an_start` and `in_or` are very close for most cases, the next check is whether `an_start` can be used when `in_or` is missing or invalid.

In [ ]:
#checking for probable nulls where in_or can use an_start

si_an.selectExpr(
    "SUM(CASE WHEN an_start IS NOT NULL AND in_or IS NULL AND out_or IS NOT NULL THEN 1 ELSE 0 END) AS only_in_or_nulls"
).show()

# si_an.filter("an_start is not null and an_stop is not null and out_or is not null and in_or is null").select(
#     "log_id","an_start","in_or","out_or","an_stop","primary_procedure_nm")

Only **10 cases** have a missing `in_or` while both `an_start` and `out_or` are available.

Since this is a very small number of records, no values are filled at this stage.

For the remaining cases with missing `in_or`, `out_or` is also missing, so an OR duration still cannot be calculated.

# Finding and reviewing outliers

## 9. Comparing OR and anesthesia duration

`or_minus_anes = or_duration_min - anes_duration_min`

This difference is used to identify cases where the OR and anesthesia durations do not agree.

- **Positive value** → OR duration is longer.
- **Negative value** → anesthesia duration is longer.
- **Close to zero** → both durations are similar.

In [ ]:
si_chk = si_an.withColumn(
    "or_minus_anes", expr("or_duration_min - anes_duration_min"))

si_chk.selectExpr(
    "COUNT(or_minus_anes) AS rows_checked",
    "percentile(or_minus_anes, 0.01) AS p01",
    "percentile(or_minus_anes, 0.25) AS p25",
    "percentile(or_minus_anes, 0.50) AS p50",
    "percentile(or_minus_anes, 0.75) AS p75",
    "percentile(or_minus_anes, 0.99) AS p99",
    "MIN(or_minus_anes) AS min", "MAX(or_minus_anes) AS max"
).show()

### Reading this output

- **p25 to p75: -10 to -6 min** → the middle 50% of cases fall within this range.
- **p50: -7 min** → in a typical case, anesthesia duration is about 7 minutes longer than OR duration.
- **p99: +4 min** → 99% of cases have `or_minus_anes` at or below 4 minutes.
- **p01: -33 min** → only 1% of cases have a difference below -33 minutes.
- **min: -2,383 min** → an unusually long anesthesia duration compared with OR duration.
- **max: +1,440 min** → exactly 24 hours, suggesting a possible timestamp/date issue.

Most cases are concentrated around **-7 minutes**. The extreme values on both ends need further review.

In [ ]:
a=si_chk.orderBy(col("or_minus_anes").desc(),col("or_duration_min").desc()).select(
    "log_id","in_or","out_or","an_start","an_stop","or_duration_min",
    "anes_duration_min","or_minus_anes","primary_procedure_nm"
)
display(a.limit(10))

## 10. Repair 1 — `out_or` recorded one day late

One case has an anesthesia duration of **235 minutes**, while the OR duration is **1,675 minutes**. The difference suggests that `out_or` was recorded one day late.

**Fix:** subtract one day from `out_or` when `or_duration_min > 1440`.

There are **8 affected records**. Five have anesthesia durations that support the one-day offset pattern, while the other three do not have anesthesia data available for comparison.

All repaired records are tracked using `out_or_repaired_flag`.

One record (`8338d2b6d17feea0`) still has an unusually long OR duration of **1,367 minutes** compared with **47 minutes** of anesthesia duration. Since it does not meet the `> 1440` repair rule and the timestamps do not show a clear repair pattern, this record is excluded rather than modified.

In [ ]:
si_r1 = (si_chk
    .withColumn("out_or_repaired_flag", expr("coalesce(or_duration_min > 1440, false)"))
    .withColumn("out_or", expr("CASE WHEN out_or_repaired_flag=true THEN timestampadd(Day, -1, out_or) ELSE out_or END"))
    .withColumn("or_duration_min", expr("timestampdiff(Minute, in_or, out_or)"))
    .withColumn("or_minus_anes" ,expr("or_duration_min - anes_duration_min")))

si_r1.filter("out_or_repaired_flag").selectExpr(
    "log_id","or_duration_min","anes_duration_min","primary_procedure_nm"
).orderBy("or_duration_min").show(truncate=False)

## 11. Repair 2 — negative duration

The negative OR duration comes from an incorrect `in_or` timestamp.

The earlier comparison showed a median gap of **0 minutes** between `an_start` and `in_or`, so `an_start` is used to replace the incorrect `in_or`.

**After repair:** OR duration is **110 minutes**, compared with **121 minutes** of anesthesia duration, giving an `or_minus_anes` of **-11 minutes**.

This falls close to the normal range observed between the two durations.

In [ ]:
si_r2 = (si_r1
    .withColumn("in_or_repaired_flag",expr("coalesce(or_duration_min <=0,false)"))
    .withColumn("in_or",expr("case when in_or_repaired_flag then an_start else in_or end"))
    .withColumn("or_duration_min", expr("timestampdiff(Minute, in_or, out_or)"))
    .withColumn("or_minus_anes" ,expr("or_duration_min - anes_duration_min")))

si_r2.filter("in_or_repaired_flag").selectExpr(
    "log_id","or_duration_min","anes_duration_min","primary_procedure_nm"
).orderBy("or_duration_min").show(truncate=False)

## 12. Repair 3 — reviewing large `in_or` gaps

To find remaining `in_or` outliers, I filtered cases where:

- `or_duration_min` is more than **3 times** `anes_duration_min`
- `start_gap` between `in_or` and `an_start` is greater than **120 minutes**
- `anes_duration_min` is at least **60 minutes**

The earlier analysis showed that **99% of cases have a start gap within 10 minutes**, so a gap above 120 minutes is well outside the normal range. The 60-minute anesthesia threshold also helps avoid using very short anesthesia records as the basis for a repair.

**Fix:** replace `in_or` with `an_start` for the confirmed cases and track the change using `in_or_repaired_flag`.

**Result:** 4 cases repaired. After the repair, the OR durations are much closer to the corresponding anesthesia durations.

In [ ]:
si_check = (si_r2
    .filter(col("or_duration_min") > 3* col("anes_duration_min"))
    .filter(col("anes_duration_min") >= 60)
    .withColumn("start_gap",expr("abs(timestampdiff(MINUTE, in_or, an_start))"))
    .filter(col("start_gap")>120)
    .withColumn("end_gap",expr("abs(timestampdiff(MINUTE, out_or, an_stop))"))
    .select("log_id","start_gap","end_gap","or_minus_anes","or_duration_min",
    "anes_duration_min","an_start","in_or","out_or","an_stop","primary_procedure_nm")
    .orderBy(col("or_duration_min").desc()))
display(si_check)

In [ ]:
exclude_logid = ["8338d2b6d17feea0"] # unresolved timestamp outlier.
in_or_repair_ids = ["6c4d7607dac1f718","15d7a340edff847f","a361a18022b74d51", "c2b283c60bf6bbcb","2713cad9cf971f24"]

si_r3 = (si_r2.withColumn("in_or_repaired_flag", col("log_id").isin(in_or_repair_ids))
    .withColumn("in_or", expr("case when in_or_repaired_flag then an_start else in_or end"))
    .withColumn("or_duration_min", expr("timestampdiff(Minute, in_or, out_or)"))
    .withColumn("or_minus_anes", expr("or_duration_min - anes_duration_min"))
    .filter(~col("log_id").isin(exclude_logid)))

si_r3.filter("in_or_repaired_flag").selectExpr(
    "log_id","in_or","out_or","or_duration_min","anes_duration_min",
    "or_duration_min - anes_duration_min AS or_minus_anes"
).show(truncate=False)

`2713cad9cf971f24`, repaired in Step 11, is included again in `in_or_repair_ids` so the `in_or_repaired_flag` remains `true` when the flag column is rebuilt in this step.

### 13. Final check after timestamp repairs

- **8** `out_or` records repaired
- **5** `in_or` records repaired
- **57,861** records have an OR duration
- no non-positive OR durations remain
- OR durations range from **3 to 1,399 minutes**
- **508** cases have OR durations above 720 minutes

The timestamp outliers have been reviewed and the required repairs are complete. No further timestamp changes are made.

In [ ]:
si_r3.selectExpr(
    "COUNT(*) AS n_rows",
    "SUM(CASE WHEN out_or_repaired_flag THEN 1 ELSE 0 END) AS out_or_repairs",
    "SUM(CASE WHEN in_or_repaired_flag  THEN 1 ELSE 0 END) AS in_or_repairs",
    "COUNT(or_duration_min) AS n_durations",
    "MIN(or_duration_min) AS min_dur",
    "MAX(or_duration_min) AS max_dur",
    "SUM(CASE WHEN or_duration_min <= 0 THEN 1 ELSE 0 END) AS non_positive",
    "SUM(CASE WHEN or_duration_min > 720 THEN 1 ELSE 0 END) AS over_720"
).show(vertical=True)

## 15. Flagging compound procedure names

Some procedure names contain `AND` or `OR`, indicating that more than one procedure may be recorded in the same name.

`is_compound_name` flags these cases using word boundaries so `AND` or `OR` are matched as standalone words rather than as part of another word.

These names are flagged rather than merged because combining them with individual procedures could affect the expected-duration baseline.

**Result:** 207 distinct compound names across 9,015 cases.

In [ ]:
from pyspark.sql.functions import upper, regexp_replace

si_named = (si_r3
    .withColumn("procedure_nm",
        upper(trim(regexp_replace(col("primary_procedure_nm"), r"\s+", " "))))
    .withColumn("is_compound_name",
        expr("procedure_nm RLIKE '\\\\b(AND|OR)\\\\b'")))

si_named.selectExpr(
    "COUNT(DISTINCT primary_procedure_nm) AS raw_names",
    "COUNT(DISTINCT procedure_nm) AS clean_names",
    "SUM(CASE WHEN is_compound_name THEN 1 ELSE 0 END) AS compound_cases",
    "COUNT(DISTINCT CASE WHEN is_compound_name THEN procedure_nm END) AS compound_names"
).show()

### Checking procedure name variants

procedure names may still differ only because of punctuation. To check this, I created a fingerprint by removing all characters except letters and numbers, then compared the number of distinct procedure names with the number of distinct fingerprints.

**Result:** 1,767 procedure names produced 1,765 fingerprints, leaving **2 potential matches** to review.

In [ ]:
si_fp = si_named.withColumn("fp", regexp_replace(col("procedure_nm"), "[^A-Z0-9]", ""))

si_fp.selectExpr(
    "COUNT(DISTINCT procedure_nm) AS names",
    "COUNT(DISTINCT fp)           AS fingerprints"
).show()

In [ ]:
from pyspark.sql.functions import collect_set

(si_fp.groupBy("fp").agg(collect_set("procedure_nm").alias("variants"))
      .filter("size(variants) > 1")
      .select("variants").show(50, truncate=False))

The fingerprint check found two groups where the procedure names differ only because of formatting or encoding.

- `CRANIOPLASTY FOR CRANIAL DEFECT` has two variants caused by trailing non-standard characters/encoding.
- `RESECTION TEMPORAL BONE, EXTERNAL APPROACH` and `RESECTION, TEMPORAL BONE, EXTERNAL APPROACH` differ only by a comma.

Both groups were reviewed and confirmed to represent the same procedure, so they are standardized to a single name.

In [ ]:
si_named = si_named.withColumn("procedure_nm", expr('''CASE
      WHEN procedure_nm LIKE 'CRANIOPLASTY FOR CRANIAL DEFECT%'
           THEN 'CRANIOPLASTY FOR CRANIAL DEFECT'
      WHEN procedure_nm = 'RESECTION TEMPORAL BONE, EXTERNAL APPROACH'
           THEN 'RESECTION, TEMPORAL BONE, EXTERNAL APPROACH'
      ELSE procedure_nm
    END'''))

si_named.selectExpr(
    "COUNT(DISTINCT primary_procedure_nm) AS raw_names",
    "COUNT(DISTINCT procedure_nm)         AS clean_names",
    "COUNT(*) AS n_rows"
).show()

## 15. Typing and derived attributes

`birth_date` contains patient age rather than a date, so it is renamed to `age_years` and cast to an integer.

`los` represent length of stay in days is cast to `los_days` as a numeric field, `asa_rating_c` is cast to integer, and `start_hour` is derived from the repaired `in_or` timestamp.`icu_admit` is created from `icu_admin_flag`, with missing values treated as `False`.

Some source columns remain unchanged because they are not needed for the current analysis. `height` contains free-text values, while `weight` is stored in ounces.

In [ ]:
from pyspark.sql.functions import hour, coalesce, lit

si_typed = (si_named
    .withColumnRenamed("birth_date", "age_years")
    .withColumn("age_years",    col("age_years").cast("int"))
    .withColumn("los_days",     col("los").cast("double"))
    .withColumn("asa_rating_c", col("asa_rating_c").cast("int"))
    .withColumn("icu_admit",    coalesce(col("icu_admin_flag") == "Yes", lit(False)))
    .withColumn("start_hour",   hour(col("in_or"))))

si_typed.selectExpr(
    "SUM(CASE WHEN age_years IS NULL THEN 1 ELSE 0 END) AS age_null",
    "SUM(CASE WHEN los_days IS NULL THEN 1 ELSE 0 END) AS los_null",
    "SUM(CASE WHEN asa_rating_c IS NULL THEN 1 ELSE 0 END) AS asa_null",
    "SUM(CASE WHEN procedure_nm IS NULL THEN 1 ELSE 0 END) AS procedure_null",
    "MIN(age_years) AS age_min", "MAX(age_years) AS age_max",
    "MIN(start_hour) AS hr_min", "MAX(start_hour) AS hr_max"
).show(vertical=True)

### Reading this output

- `age_years` has no nulls and ranges from **17 to 90**.
- `los_days` has **14 nulls**.
- `asa_rating_c` has **6,809 nulls** and is left as missing rather than imputed.
- `procedure_nm` has **6 nulls**.
- `start_hour` ranges from **0 to 23**, confirming that cases occur across all hours of the day.

## 16. Checking the remaining categorical columns

The remaining categorical columns are reviewed using value counts to check for inconsistent labels, spelling, casing, or formatting that could split the same category into multiple values.

**Result:** all six columns are consistent and no further standardization is needed.

In [ ]:
for c in ["sex", "asa_rating","primary_anes_type_nm", "patient_class_group", "patient_class_nm", "disch_disp"]:
    print(f"--- {c} ---")
    si_typed.groupBy(c).count().orderBy(col("count").desc()).show(20, truncate=False)

## 17. Final validation and Silver write

Before writing the final Silver table, the row count and schema are checked to make sure the transformations have not introduced unexpected changes.

In [ ]:
print("Rows:", si_typed.count())
print("Columns:", len(si_typed.columns))

assert si_typed.count() == 64353
assert all(c == c.lower() for c in si_typed.columns)

In [ ]:
si_typed.write.mode("overwrite").option("overwriteSchema","true").format("delta").saveAsTable("silver.patient_information")

In [ ]:
display(spark.sql("DESCRIBE TABLE silver.patient_information"))

# Silver layer summary

### Row reconciliation

| stage | rows |
|---|---:|
| raw `patient_information` | 65,728 |
| after trim and duplicate removal | 64,362 |
| after resolving duplicate `log_id` | 64,354 |
| after excluding one unresolved timestamp outlier | **64,353** |
| records with OR duration | **57,861** |

### What was completed

- Cleaned and standardized column names, text values, data types, and categorical fields.
- Removed duplicate records and resolved duplicate `log_id` values.
- Parsed and validated OR and anesthesia timestamps, using the anesthesia timestamps to investigate unusual OR durations.
- Repaired **8 `out_or`** and **5 `in_or`** timestamps and excluded **1 unresolved timestamp outlier**. No non-positive OR durations remain.
- Standardized procedure names from **1,768 raw names to 1,765 final names** and flagged **207 compound names across 9,015 cases (~14%)**.
- Created the required derived fields and retained data-quality and repair flags for downstream analysis.

### Final Silver table

`silver.patient_information` contains **64,353 rows and 39 columns**.

The Silver layer now contains the cleaned and standardized case-level data. Analytical cohort filters are left for the Gold layer.

**Next:** `03_gold` builds the procedure-level baseline and case-level analytical tables from `silver.patient_information`.
